In [6]:
!pip install --upgrade  openai anthropic google-generativeai together pandas numpy


[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [9]:
pip install --upgrade pip

  Using cached pip-25.3-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 23.0.1
    Not uninstalling pip at /usr/local/lib/python3.10/site-packages, outside environment /root/venv
    Can't uninstall 'pip'. No files were found to uninstall.

[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
"""
ABLATION STUDY - STRUCTURED FORMAT WITH FULL METHODOLOGY (REVISED)
===================================================================
Tests feature importance using structured format WITH complete survey methodology.

This version:
- Uses clean structured format (bullet points, headers)
- Includes FULL survey methodology (matching GEN_DATA)
- Maintains climate-specific framing
- Directly comparable to narrative format studies

KEY CHANGES FROM ORIGINAL STRUCTURED VERSION:
✓ Added complete personal willingness question
✓ Added response options (Yes, No, Don't Know, Refused)
✓ Added missing data handling note
✓ Added second-order belief question setup
✓ Now matches GEN_DATA methodology exactly

FEATURES:
1. ✓ Checkpoint/resume system
2. ✓ Keep-alive thread
3. ✓ Retry logic with exponential backoff
4. ✓ All 10 conditions (7 ablations + 3 counterfactuals)
5. ✓ Progress tracking
6. ✓ Forbidden string detection
"""

import os, re, time, json, threading
from datetime import datetime
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# API imports
from openai import OpenAI
import anthropic
import google.generativeai as genai
from together import Together

print("="*80)
print("ABLATION STUDY - STRUCTURED FORMAT WITH FULL METHODOLOGY (REVISED)")
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)

# ================================================================
# Keep-alive thread
# ================================================================

def keep_alive():
    while True:
        if time.time() % 300 < 60:
            print(f"\n⏱️  [{datetime.now().strftime('%H:%M:%S')}] Session alive", flush=True)
        else:
            print(".", end="", flush=True)
        time.sleep(60)

keep_alive_thread = threading.Thread(target=keep_alive, daemon=True)
keep_alive_thread.start()
print("✓ Keep-alive thread started\n")

# ================================================================
# Configuration
# ================================================================

CHECKPOINT_FILE = "ablation_structured_revised_checkpoint.json"
RAW_RESULTS_FILE = "ablation_structured_revised_raw_results.csv"
DATA_FILE = "data_final.csv"

# API Configuration
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
CLAUDE_API_KEY = os.getenv("CLAUDE_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
LLAMA_API_KEY = os.getenv("LLAMA_API_KEY")

# Model configurations
MODELS = {
    "gpt": "gpt-4o-mini",
    "claude": "claude-3-5-haiku-20241022",
    "gemini": "gemini-2.5-flash",
    "llama": "meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8"
}

# System instruction
SYSTEM_INSTRUCTION = """You are a prediction assistant making estimates based ONLY on the information provided in this specific prompt.

CRITICAL INSTRUCTIONS:
1. Do NOT cite, reference, or mention ANY research papers, academic studies, surveys, or authors
2. Do NOT use any memorized data, statistics, or percentages from your training about climate change opinions, pluralistic ignorance, or survey results
3. Treat this as a completely NOVEL scenario - ignore any similar studies you may have seen during training
4. Do NOT reference 'research shows', 'studies indicate', 'surveys have found', or similar phrases
5. Base your estimate ONLY on:
   - General reasoning about human psychology and behavior
   - The specific information provided in this prompt
   - First principles about how people form beliefs about others

Your task is to predict what percentage people THINK others believe (second-order belief), not what people actually believe (first-order belief). This is a prediction task requiring general reasoning, not recall of specific research findings.

Respond with ONLY a JSON object containing a single number between 0 and 100 with one decimal place: {"prediction": XX.X}

Do not include any explanation, reasoning, or text - only the JSON."""

# Condition order
condition_order = [
    'full',
    'no_econ',
    'no_religion',
    'no_demo',
    'no_climate',
    'no_own_willingness',
    'country_only',
    'cf_gdp_flip',
    'cf_name_mismatch',
    'cf_willingness_flip'
]

# ================================================================
# Helper Functions
# ================================================================

def as_num(x, nd=1):
    if pd.isna(x): 
        return None
    try:
        return round(float(x), nd)
    except:
        return None

def as_pct(x):
    if pd.isna(x): 
        return None
    try:
        v = float(x)
    except:
        return None
    if 0 <= v <= 1:
        return v * 100.0
    return v

def fmt_pct(x):
    v = as_pct(x)
    return f"{v:.1f}%" if v is not None else "N/A"

def fmt_num(x, nd=1):
    v = as_num(x, nd)
    return f"{v:.{nd}f}" if v is not None else "N/A"

def extract_number_0_100(text):
    if not isinstance(text, str):
        return None
    
    try:
        data = json.loads(text)
        if isinstance(data, dict):
            for key in ["prediction", "estimate", "value", "number", "percentage"]:
                if key in data:
                    val = float(data[key])
                    return max(0.0, min(100.0, val))
        elif isinstance(data, (int, float)):
            val = float(data)
            return max(0.0, min(100.0, val))
    except:
        pass
    
    m = re.search(r"(\d+(?:\.\d+)?)", text)
    if not m:
        return None
    val = float(m.group(1))
    return max(0.0, min(100.0, val))

def contains_forbidden_strings(text):
    if not isinstance(text, str):
        return False
    
    text_lower = text.lower()
    forbidden = [
        "andré", "andre", "et al", "et. al", "et.al",
        "doi", "http://", "https://",
        "paper", "study", "research",
        "published", "journal", "article"
    ]
    
    return any(term in text_lower for term in forbidden)

# ================================================================
# Load Data
# ================================================================

print("\n1. Loading data...")
if not Path(DATA_FILE).exists():
    print(f"   ❌ Data file not found: {DATA_FILE}")
    exit(1)

df = pd.read_csv(DATA_FILE)
print(f"   ✓ Loaded {len(df)} countries")

# Column identification
OWN_LESS_COL = None
for col_name in ["mean_own_willingness_less", "mean_own_willigness_less"]:
    if col_name in df.columns:
        OWN_LESS_COL = col_name
        break

TEMP_COL = None
for col_name in ["temp_mean", "temp_mean_2010_2019"]:
    if col_name in df.columns:
        TEMP_COL = col_name
        break

print(f"   ✓ Temperature column: {TEMP_COL}")
print(f"   ✓ Willingness_less column: {OWN_LESS_COL}")

# ================================================================
# REVISED Structured Prompt Builders WITH FULL METHODOLOGY
# ================================================================

def build_ablated_prompts_structured(row, all_countries_df):
    """
    Build all ablation versions using structured format WITH full methodology.
    Now includes complete survey setup matching GEN_DATA.
    """
    country = row["countrynew"]
    prompts = {}
    
    # ================================================================
    # FULL METHODOLOGY SECTION (used in all prompts)
    # ================================================================
    
    survey_methodology = f"""Survey Methodology:
This data comes from a nationally representative survey with a probability-based sample of approximately 1,000 residents aged 15 and above in {country}.

First Question (Personal Willingness):
Respondents were asked: "Would you be willing to contribute 1% of your household income every month to fight global warming? This would mean that you would contribute 1 for every 100 of this income."
- Response options: Yes, No, (Don't Know), (Refused)
- Note: Don't know and refused were coded as missing data

Second Question (Belief About Others):
Respondents were then asked how many respondents in {country} they think are willing to contribute at least 1% of their household income every month to fight global warming.
- Response format: between 0% and 100%, (Don't know), (Refused)"""
    
    # ================================================================
    # BUILD CONTEXT SECTIONS
    # ================================================================
    
    demographics_full = f"""Demographics:
- Average age: {fmt_num(row.get('mean_age'), 1)} years
- Tertiary education: {fmt_pct(row.get('mean_edu'))} completed
- Religion importance: {fmt_pct(row.get('mean_religion'))} say it's important in daily life"""

    demographics_no_religion = f"""Demographics:
- Average age: {fmt_num(row.get('mean_age'), 1)} years
- Tertiary education: {fmt_pct(row.get('mean_edu'))} completed"""

    economy_full = f"""Economy:
- GDP per capita (PPP, 2021): ${fmt_num(row.get('gdp_capita_2021'), 0)}
- Top 1% income share: {fmt_pct(row.get('top1pct_income'))}
- Top 1% wealth share: {fmt_pct(row.get('top1pct_wealth'))}
- Human Development Index (2021): {fmt_num(row.get('hdi_2021'), 3)}"""

    economy_gdp_only = f"""Economy:
- GDP per capita (PPP, 2021): ${fmt_num(row.get('gdp_capita_2021'), 0)}"""

    religion_only = f"""Religion:
- Religion importance: {fmt_pct(row.get('mean_religion'))} say it's important in daily life"""

    climate_data = f"""Climate:
- Average temperature (2010-2019): {fmt_num(row.get(TEMP_COL), 1)}°C"""

    actual_willingness = f"""Personal Willingness (Actual Survey Data):
- {fmt_pct(row.get('mean_own_willingness'))} said they would contribute 1% of their household income monthly
- {fmt_pct(row.get(OWN_LESS_COL)) if OWN_LESS_COL else 'N/A'} said they would contribute a smaller amount"""

    task_instruction = f"""Your Task:
Based on the country and data provided above, estimate what respondents in {country} on average thought about how many OTHER respondents in {country} are willing to contribute at least 1% of their household income every month to fight global warming.

Note: You are estimating people's BELIEFS about others' willingness, not the actual willingness itself.

Respond with a single number between 0 and 100 with one decimal place."""
    
    # ================================================================
    # CONDITION 1: FULL (BASELINE)
    # ================================================================
    
    prompts['full'] = f"""Country: {country}

{demographics_full}

{economy_full}

{climate_data}

{survey_methodology}

{actual_willingness}

{task_instruction}"""
    
    # ================================================================
    # CONDITION 2: NO ECONOMIC INDICATORS
    # ================================================================
    
    prompts['no_econ'] = f"""Country: {country}

{demographics_full}

{climate_data}

{survey_methodology}

{actual_willingness}

{task_instruction}"""
    
    # ================================================================
    # CONDITION 3: NO RELIGION
    # ================================================================
    
    prompts['no_religion'] = f"""Country: {country}

{demographics_no_religion}

{economy_full}

{climate_data}

{survey_methodology}

{actual_willingness}

{task_instruction}"""
    
    # ================================================================
    # CONDITION 4: NO DEMOGRAPHICS
    # ================================================================
    
    prompts['no_demo'] = f"""Country: {country}

{economy_gdp_only}

{religion_only}

{climate_data}

{survey_methodology}

{actual_willingness}

{task_instruction}"""
    
    # ================================================================
    # CONDITION 5: NO CLIMATE
    # ================================================================
    
    prompts['no_climate'] = f"""Country: {country}

{demographics_full}

{economy_full}

{survey_methodology}

{actual_willingness}

{task_instruction}"""
    
    # ================================================================
    # CONDITION 6: NO OWN WILLINGNESS
    # ================================================================
    
    prompts['no_own_willingness'] = f"""Country: {country}

{demographics_full}

{economy_full}

{climate_data}

{survey_methodology}

{task_instruction}"""
    
    # ================================================================
    # CONDITION 7: COUNTRY ONLY
    # ================================================================
    
    task_minimal = f"""Your Task:
Based only on the country name, estimate what respondents in {country} on average thought about how many OTHER respondents in {country} are willing to contribute at least 1% of their household income every month to fight global warming.

Note: You are estimating people's BELIEFS about others' willingness, not the actual willingness itself.

Respond with a single number between 0 and 100 with one decimal place."""

    prompts['country_only'] = f"""Country: {country}

{survey_methodology}

{task_minimal}"""
    
    # ================================================================
    # COUNTERFACTUAL 1: GDP FLIP
    # ================================================================
    
    gdp = row.get('gdp_capita_2021', 0)
    is_rich = gdp > 20000
    cf_gdp_str = "$2,000" if is_rich else "$65,000"
    
    economy_gdp_flipped = f"""Economy:
- GDP per capita (PPP, 2021): {cf_gdp_str}
- Top 1% income share: {fmt_pct(row.get('top1pct_income'))}
- Top 1% wealth share: {fmt_pct(row.get('top1pct_wealth'))}
- Human Development Index (2021): {fmt_num(row.get('hdi_2021'), 3)}"""
    
    prompts['cf_gdp_flip'] = f"""Country: {country}

{demographics_full}

{economy_gdp_flipped}

{climate_data}

{survey_methodology}

{actual_willingness}

{task_instruction}"""
    
    # ================================================================
    # COUNTERFACTUAL 2: NAME-DATA MISMATCH
    # ================================================================
    
    all_countries_sorted = sorted(all_countries_df['countrynew'].unique())
    country_idx = all_countries_sorted.index(country) if country in all_countries_sorted else 0
    
    if is_rich:
        poor_countries = all_countries_df[all_countries_df['gdp_capita_2021'] < 5000]['countrynew'].tolist()
        cf_name = sorted(poor_countries)[country_idx % len(poor_countries)] if poor_countries else "Chad"
    else:
        rich_countries = all_countries_df[all_countries_df['gdp_capita_2021'] > 40000]['countrynew'].tolist()
        cf_name = sorted(rich_countries)[country_idx % len(rich_countries)] if rich_countries else "Norway"
    
    survey_methodology_cf = f"""Survey Methodology:
This data comes from a nationally representative survey with a probability-based sample of approximately 1,000 residents aged 15 and above in {cf_name}.

First Question (Personal Willingness):
Respondents were asked: "Would you be willing to contribute 1% of your household income every month to fight global warming? This would mean that you would contribute 1 for every 100 of this income."
- Response options: Yes, No, (Don't Know), (Refused)
- Note: Don't know and refused were coded as missing data

Second Question (Belief About Others):
Respondents were then asked how many respondents in {cf_name} they think are willing to contribute at least 1% of their household income every month to fight global warming.
- Response format: between 0% and 100%, (Don't know), (Refused)"""

    task_instruction_cf = f"""Your Task:
Based on the country and data provided above, estimate what respondents in {cf_name} on average thought about how many OTHER respondents in {cf_name} are willing to contribute at least 1% of their household income every month to fight global warming.

Note: You are estimating people's BELIEFS about others' willingness, not the actual willingness itself.

Respond with a single number between 0 and 100 with one decimal place."""
    
    prompts['cf_name_mismatch'] = f"""Country: {cf_name}

{demographics_full}

{economy_full}

{climate_data}

{survey_methodology_cf}

{actual_willingness}

{task_instruction_cf}"""
    
    # ================================================================
    # COUNTERFACTUAL 3: WILLINGNESS FLIP
    # ================================================================
    
    own_willingness = row.get('mean_own_willingness', 0)
    is_high_willingness = own_willingness > 50
    
    if is_high_willingness:
        cf_own_main = "15.0%"
        cf_own_less = "10.0%"
    else:
        cf_own_main = "75.0%"
        cf_own_less = "15.0%"
    
    actual_willingness_flipped = f"""Personal Willingness (Actual Survey Data):
- {cf_own_main} said they would contribute 1% of their household income monthly
- {cf_own_less} said they would contribute a smaller amount"""
    
    prompts['cf_willingness_flip'] = f"""Country: {country}

{demographics_full}

{economy_full}

{climate_data}

{survey_methodology}

{actual_willingness_flipped}

{task_instruction}"""
    
    return prompts

# ================================================================
# API Functions
# ================================================================

def call_gpt(prompt, model="gpt-4o-mini", max_retries=3):
    if not OPENAI_API_KEY:
        return None
    client = OpenAI(api_key=OPENAI_API_KEY)
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": SYSTEM_INSTRUCTION},
                    {"role": "user", "content": prompt}
                ],
                temperature=0,
                response_format={"type": "json_object"},
            )
            content = response.choices[0].message.content
            if contains_forbidden_strings(content):
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                    continue
                return None
            return extract_number_0_100(content)
        except Exception as e:
            print(f"❌ GPT attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
    return None

def call_claude(prompt, model="claude-3-5-haiku-20241022", max_retries=5):
    if not CLAUDE_API_KEY:
        return None
    client = anthropic.Anthropic(api_key=CLAUDE_API_KEY)
    for attempt in range(max_retries):
        try:
            response = client.messages.create(
                model=model,
                max_tokens=100,
                temperature=0,
                system=SYSTEM_INSTRUCTION,
                messages=[{"role": "user", "content": prompt}],
                tools=[],
            )
            content = response.content[0].text
            if contains_forbidden_strings(content):
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                    continue
                return None
            return extract_number_0_100(content)
        except anthropic.RateLimitError:
            wait_time = (2 ** attempt) * 2
            print(f"⚠️  Claude rate limit, waiting {wait_time}s...")
            if attempt < max_retries - 1:
                time.sleep(wait_time)
        except Exception as e:
            print(f"❌ Claude attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
    return None

def call_gemini(prompt, model="gemini-2.5-flash", max_retries=3):
    if not GEMINI_API_KEY:
        return None
    genai.configure(api_key=GEMINI_API_KEY)
    model_obj = genai.GenerativeModel(model)
    full_prompt = f"{SYSTEM_INSTRUCTION}\n\n{prompt}"
    for attempt in range(max_retries):
        try:
            response = model_obj.generate_content(
                full_prompt,
                generation_config=genai.types.GenerationConfig(temperature=0)
            )
            content = response.text
            if contains_forbidden_strings(content):
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                    continue
                return None
            return extract_number_0_100(content)
        except Exception as e:
            print(f"❌ Gemini attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
    return None

def call_llama(prompt, model="meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8", max_retries=3):
    if not LLAMA_API_KEY:
        return None
    client = Together(api_key=LLAMA_API_KEY)
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": SYSTEM_INSTRUCTION},
                    {"role": "user", "content": prompt}
                ],
                temperature=0,
            )
            content = response.choices[0].message.content
            if contains_forbidden_strings(content):
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                    continue
                return None
            return extract_number_0_100(content)
        except Exception as e:
            print(f"❌ Llama attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
    return None

# ================================================================
# Checkpoint Management
# ================================================================

def load_checkpoint():
    if Path(CHECKPOINT_FILE).exists():
        with open(CHECKPOINT_FILE, 'r') as f:
            return json.load(f)
    return {"completed": []}

def save_checkpoint(checkpoint):
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(checkpoint, f)

def is_completed(checkpoint, country, condition, model):
    key = f"{country}|{condition}|{model}"
    return key in checkpoint["completed"]

def mark_completed(checkpoint, country, condition, model):
    key = f"{country}|{condition}|{model}"
    if key not in checkpoint["completed"]:
        checkpoint["completed"].append(key)

# ================================================================
# Main Study
# ================================================================

def run_ablation_study():
    print("\n2. Loading checkpoint...")
    checkpoint = load_checkpoint()
    print(f"   ✓ {len(checkpoint['completed'])} tasks already completed")
    
    if Path(RAW_RESULTS_FILE).exists():
        results_df = pd.read_csv(RAW_RESULTS_FILE)
        print(f"   ✓ Loaded existing results: {len(results_df)} rows")
    else:
        results_df = pd.DataFrame()
        print("   ✓ Starting fresh results file")
    
    available_models = []
    for model in MODELS.keys():
        if model == "gpt" and OPENAI_API_KEY:
            available_models.append(model)
        elif model == "claude" and CLAUDE_API_KEY:
            available_models.append(model)
        elif model == "gemini" and GEMINI_API_KEY:
            available_models.append(model)
        elif model == "llama" and LLAMA_API_KEY:
            available_models.append(model)
    
    total_tasks = len(df) * len(condition_order) * len(available_models)
    completed_tasks = len(checkpoint['completed'])
    
    print(f"\n3. Running ablation study...")
    print(f"   Total tasks: {total_tasks}")
    print(f"   Remaining: {total_tasks - completed_tasks}")
    print(f"   Progress: {100 * completed_tasks / total_tasks:.1f}%\n")
    
    results = []
    start_time = time.time()
    
    for i, (idx, row) in enumerate(df.iterrows()):
        country = row['countrynew']
        print(f"\n[{i+1}/{len(df)}] Processing: {country}")
        
        all_prompts = build_ablated_prompts_structured(row, df)
        
        for condition in condition_order:
            prompt = all_prompts[condition]
            
            for model_name, model_id in MODELS.items():
                if is_completed(checkpoint, country, condition, model_name):
                    continue
                
                if model_name not in available_models:
                    continue
                
                print(f"  [{condition:20s}] [{model_name:7s}] ", end="", flush=True)
                
                if model_name == "gpt":
                    prediction = call_gpt(prompt, model_id)
                elif model_name == "claude":
                    prediction = call_claude(prompt, model_id)
                elif model_name == "gemini":
                    prediction = call_gemini(prompt, model_id)
                elif model_name == "llama":
                    prediction = call_llama(prompt, model_id)
                else:
                    prediction = None
                
                if prediction is not None:
                    print(f"✓ {prediction:.1f}")
                else:
                    print(f"✗ Failed")
                
                results.append({
                    'country': country,
                    'condition': condition,
                    'model': model_name,
                    'prediction': prediction,
                    'actual_other_willingness': row.get('mean_other_willingness'),
                    'actual_own_willingness': row.get('mean_own_willingness'),
                })
                
                mark_completed(checkpoint, country, condition, model_name)
                completed_tasks += 1
                
                if completed_tasks % 10 == 0:
                    save_checkpoint(checkpoint)
                    
                    if results:
                        new_results_df = pd.DataFrame(results)
                        if len(results_df) > 0:
                            results_df = pd.concat([results_df, new_results_df], ignore_index=True)
                        else:
                            results_df = new_results_df
                        results_df.to_csv(RAW_RESULTS_FILE, index=False)
                        results = []
                    
                    elapsed = time.time() - start_time
                    rate = completed_tasks / elapsed
                    remaining = total_tasks - completed_tasks
                    eta_seconds = remaining / rate if rate > 0 else 0
                    eta_minutes = eta_seconds / 60
                    
                    print(f"\n  💾 Checkpoint saved | Progress: {100 * completed_tasks / total_tasks:.1f}% | ETA: {eta_minutes:.1f} min\n")
    
    # Final save
    if results:
        new_results_df = pd.DataFrame(results)
        if len(results_df) > 0:
            results_df = pd.concat([results_df, new_results_df], ignore_index=True)
        else:
            results_df = new_results_df
        results_df.to_csv(RAW_RESULTS_FILE, index=False)
    
    save_checkpoint(checkpoint)
    
    print("\n" + "="*80)
    print("ABLATION STUDY COMPLETE!")
    print("="*80)
    print(f"Results saved to: {RAW_RESULTS_FILE}")
    print(f"Total predictions: {len(results_df)}")
    print(f"Time elapsed: {(time.time() - start_time) / 60:.1f} minutes")
    
    return results_df

# ================================================================
# Run
# ================================================================

if __name__ == "__main__":
    try:
        results = run_ablation_study()
    except KeyboardInterrupt:
        print("\n\n⚠️  Interrupted by user")
        print("Progress has been saved. Run again to resume.")
    except Exception as e:
        print(f"\n\n❌ Error: {e}")
        print("Progress has been saved. Run again to resume.")
        raise

╭───────────────────────────────────────────── 🚀 New SDK Available ──────────────────────────────────────────────╮
│ Together Python SDK 2.0 is now available!                                                                       │
│                                                                                                                 │
│ Install the beta:                                                                                               │
│ pip install --pre together  or  uv add together --prerelease allow                                              │
│                                                                                                                 │
│ New SDK: ]8;id=288876;https://github.com/togethercomputer/together-py\https://github.com/togethercomputer/together-py]8;;\                                                        │
│ Migration guide: ]8;id=700314;https://docs.together.ai/docs/pythonv2-migration-guide\https://docs.together.ai/docs/pythonv2-migration-guide]8;;\                                         │
│                                                                                                                 │
│ This package will be maintained until January 2026.                                                             │
│ Set TOGETHER_NO_BANNER=1 to hide this message.                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⏱️  [07:55:22] Session alive
✓ 44.8
  [no_religion         ] [gpt    ] ✓ 50.0
  [no_religion         ] [claude ] ✓ 45.7

  💾 Checkpoint saved | Progress: 85.8% | ETA: 67.8 min

  [no_religion         ] [gemini ] ✓ 39.9
  [no_religion         ] [llama  ] ✓ 44.8
  [no_demo             ] [gpt    ] ✓ 50.0
  [no_demo             ] [claude ] ✓ 45.2
  [no_demo             ] [gemini ] ✓ 47.5
  [no_demo             ] [llama  ] ✓ 44.8
  [no_climate          ] [gpt    ] ✓ 75.0
  [no_climate          ] [claude ] ✓ 45.7
  [no_climate          ] [gemini ] .✓ 53.5
  [no_climate          ] [llama  ] ✓ 44.8

  💾 Checkpoint saved | Progress: 86.0% | ETA: 66.8 min

  [no_own_willingness  ] [gpt    ] ✓ 45.0
  [no_own_willingness  ] [claude ] ✓ 42.7
  [no_own_willingness  ] [gemini ] ✓ 25.0
  [no_own_willingness  ] [llama  ] ✓ 34.5
  [country_only        ] [gpt    ] ✓ 45.0
  [country_only        ] [claude ] ✓ 42.5
  [country_only        ] [gemini ] ✓ 32.5
  [country_only        ] [llama  ] ✓ 30.5
  [cf_gdp

In [ ]:
"""
FILL MISSING PREDICTIONS - REVISED STRUCTURED ABLATION
=======================================================
Identifies gaps in the revised structured ablation results and re-runs
only the missing API calls.

This script uses the EXACT SAME prompts as ablation_study_structured_climate_revised.py
to ensure consistency.

USAGE:
python fill_missing_structured_revised.py
"""

import os, re, time, json
from datetime import datetime
import pandas as pd
import numpy as np
from pathlib import Path

# API imports
from openai import OpenAI
import anthropic
import google.generativeai as genai
from together import Together

print("="*80)
print("FILLING MISSING PREDICTIONS - REVISED STRUCTURED ABLATION")
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)

# ================================================================
# Configuration
# ================================================================

RAW_RESULTS_FILE = "ablation_structured_revised_raw_results.csv"
OUTPUT_FILE = "ablation_structured_revised_raw_results_complete.csv"
DATA_FILE = "data_final.csv"

# API Configuration
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
CLAUDE_API_KEY = os.getenv("CLAUDE_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
LLAMA_API_KEY = os.getenv("LLAMA_API_KEY")

# Model configurations
MODELS = {
    "gpt": "gpt-4o-mini",
    "claude": "claude-3-5-haiku-20241022",
    "gemini": "gemini-2.5-flash",
    "llama": "meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8"
}

# System instruction
SYSTEM_INSTRUCTION = """You are a prediction assistant making estimates based ONLY on the information provided in this specific prompt.

CRITICAL INSTRUCTIONS:
1. Do NOT cite, reference, or mention ANY research papers, academic studies, surveys, or authors
2. Do NOT use any memorized data, statistics, or percentages from your training about climate change opinions, pluralistic ignorance, or survey results
3. Treat this as a completely NOVEL scenario - ignore any similar studies you may have seen during training
4. Do NOT reference 'research shows', 'studies indicate', 'surveys have found', or similar phrases
5. Base your estimate ONLY on:
   - General reasoning about human psychology and behavior
   - The specific information provided in this prompt
   - First principles about how people form beliefs about others

Your task is to predict what percentage people THINK others believe (second-order belief), not what people actually believe (first-order belief). This is a prediction task requiring general reasoning, not recall of specific research findings.

Respond with ONLY a JSON object containing a single number between 0 and 100 with one decimal place: {"prediction": XX.X}

Do not include any explanation, reasoning, or text - only the JSON."""

# Condition order
condition_order = [
    'full',
    'no_econ',
    'no_religion',
    'no_demo',
    'no_climate',
    'no_own_willingness',
    'country_only',
    'cf_gdp_flip',
    'cf_name_mismatch',
    'cf_willingness_flip'
]

# ================================================================
# Helper Functions
# ================================================================

def as_num(x, nd=1):
    if pd.isna(x): 
        return None
    try:
        return round(float(x), nd)
    except:
        return None

def as_pct(x):
    if pd.isna(x): 
        return None
    try:
        v = float(x)
    except:
        return None
    if 0 <= v <= 1:
        return v * 100.0
    return v

def fmt_pct(x):
    v = as_pct(x)
    return f"{v:.1f}%" if v is not None else "N/A"

def fmt_num(x, nd=1):
    v = as_num(x, nd)
    return f"{v:.{nd}f}" if v is not None else "N/A"

def extract_number_0_100(text):
    if not isinstance(text, str):
        return None
    
    try:
        data = json.loads(text)
        if isinstance(data, dict):
            for key in ["prediction", "estimate", "value", "number", "percentage"]:
                if key in data:
                    val = float(data[key])
                    return max(0.0, min(100.0, val))
        elif isinstance(data, (int, float)):
            val = float(data)
            return max(0.0, min(100.0, val))
    except:
        pass
    
    m = re.search(r"(\d+(?:\.\d+)?)", text)
    if not m:
        return None
    val = float(m.group(1))
    return max(0.0, min(100.0, val))

def contains_forbidden_strings(text):
    if not isinstance(text, str):
        return False
    
    text_lower = text.lower()
    forbidden = [
        "andré", "andre", "et al", "et. al", "et.al",
        "doi", "http://", "https://",
        "paper", "study", "research",
        "published", "journal", "article"
    ]
    
    return any(term in text_lower for term in forbidden)

# ================================================================
# Load Data
# ================================================================

print("\n1. Loading data...")

if not Path(DATA_FILE).exists():
    print(f"   ❌ Data file not found: {DATA_FILE}")
    exit(1)

df = pd.read_csv(DATA_FILE)
print(f"   ✓ Loaded {len(df)} countries")

# Column identification
OWN_LESS_COL = None
for col_name in ["mean_own_willingness_less", "mean_own_willigness_less"]:
    if col_name in df.columns:
        OWN_LESS_COL = col_name
        break

TEMP_COL = None
for col_name in ["temp_mean", "temp_mean_2010_2019"]:
    if col_name in df.columns:
        TEMP_COL = col_name
        break

print(f"   ✓ Temperature column: {TEMP_COL}")
print(f"   ✓ Willingness_less column: {OWN_LESS_COL}")

if not Path(RAW_RESULTS_FILE).exists():
    print(f"   ❌ Results file not found: {RAW_RESULTS_FILE}")
    print("   Run the main ablation study first!")
    exit(1)

results_df = pd.read_csv(RAW_RESULTS_FILE)
print(f"   ✓ Loaded existing results: {len(results_df)} rows")

# ================================================================
# REVISED STRUCTURED PROMPT BUILDERS - COPIED FROM MAIN SCRIPT
# ================================================================

def build_ablated_prompts_structured(row, all_countries_df):
    """
    Build all ablation versions using structured format WITH full methodology.
    EXACT COPY from ablation_study_structured_climate_revised.py
    """
    country = row["countrynew"]
    prompts = {}
    
    # ================================================================
    # FULL METHODOLOGY SECTION (used in all prompts)
    # ================================================================
    
    survey_methodology = f"""Survey Methodology:
This data comes from a nationally representative survey with a probability-based sample of approximately 1,000 residents aged 15 and above in {country}.

First Question (Personal Willingness):
Respondents were asked: "Would you be willing to contribute 1% of your household income every month to fight global warming? This would mean that you would contribute 1 for every 100 of this income."
- Response options: Yes, No, (Don't Know), (Refused)
- Note: Don't know and refused were coded as missing data

Second Question (Belief About Others):
Respondents were then asked how many respondents in {country} they think are willing to contribute at least 1% of their household income every month to fight global warming.
- Response format: between 0% and 100%, (Don't know), (Refused)"""
    
    # ================================================================
    # BUILD CONTEXT SECTIONS
    # ================================================================
    
    demographics_full = f"""Demographics:
- Average age: {fmt_num(row.get('mean_age'), 1)} years
- Tertiary education: {fmt_pct(row.get('mean_edu'))} completed
- Religion importance: {fmt_pct(row.get('mean_religion'))} say it's important in daily life"""

    demographics_no_religion = f"""Demographics:
- Average age: {fmt_num(row.get('mean_age'), 1)} years
- Tertiary education: {fmt_pct(row.get('mean_edu'))} completed"""

    economy_full = f"""Economy:
- GDP per capita (PPP, 2021): ${fmt_num(row.get('gdp_capita_2021'), 0)}
- Top 1% income share: {fmt_pct(row.get('top1pct_income'))}
- Top 1% wealth share: {fmt_pct(row.get('top1pct_wealth'))}
- Human Development Index (2021): {fmt_num(row.get('hdi_2021'), 3)}"""

    economy_gdp_only = f"""Economy:
- GDP per capita (PPP, 2021): ${fmt_num(row.get('gdp_capita_2021'), 0)}"""

    religion_only = f"""Religion:
- Religion importance: {fmt_pct(row.get('mean_religion'))} say it's important in daily life"""

    climate_data = f"""Climate:
- Average temperature (2010-2019): {fmt_num(row.get(TEMP_COL), 1)}°C"""

    actual_willingness = f"""Personal Willingness (Actual Survey Data):
- {fmt_pct(row.get('mean_own_willingness'))} said they would contribute 1% of their household income monthly
- {fmt_pct(row.get(OWN_LESS_COL)) if OWN_LESS_COL else 'N/A'} said they would contribute a smaller amount"""

    task_instruction = f"""Your Task:
Based on the country and data provided above, estimate what respondents in {country} on average thought about how many OTHER respondents in {country} are willing to contribute at least 1% of their household income every month to fight global warming.

Note: You are estimating people's BELIEFS about others' willingness, not the actual willingness itself.

Respond with a single number between 0 and 100 with one decimal place."""
    
    # ================================================================
    # CONDITION 1: FULL (BASELINE)
    # ================================================================
    
    prompts['full'] = f"""Country: {country}

{demographics_full}

{economy_full}

{climate_data}

{survey_methodology}

{actual_willingness}

{task_instruction}"""
    
    # ================================================================
    # CONDITION 2: NO ECONOMIC INDICATORS
    # ================================================================
    
    prompts['no_econ'] = f"""Country: {country}

{demographics_full}

{climate_data}

{survey_methodology}

{actual_willingness}

{task_instruction}"""
    
    # ================================================================
    # CONDITION 3: NO RELIGION
    # ================================================================
    
    prompts['no_religion'] = f"""Country: {country}

{demographics_no_religion}

{economy_full}

{climate_data}

{survey_methodology}

{actual_willingness}

{task_instruction}"""
    
    # ================================================================
    # CONDITION 4: NO DEMOGRAPHICS
    # ================================================================
    
    prompts['no_demo'] = f"""Country: {country}

{economy_gdp_only}

{religion_only}

{climate_data}

{survey_methodology}

{actual_willingness}

{task_instruction}"""
    
    # ================================================================
    # CONDITION 5: NO CLIMATE
    # ================================================================
    
    prompts['no_climate'] = f"""Country: {country}

{demographics_full}

{economy_full}

{survey_methodology}

{actual_willingness}

{task_instruction}"""
    
    # ================================================================
    # CONDITION 6: NO OWN WILLINGNESS
    # ================================================================
    
    prompts['no_own_willingness'] = f"""Country: {country}

{demographics_full}

{economy_full}

{climate_data}

{survey_methodology}

{task_instruction}"""
    
    # ================================================================
    # CONDITION 7: COUNTRY ONLY
    # ================================================================
    
    task_minimal = f"""Your Task:
Based only on the country name, estimate what respondents in {country} on average thought about how many OTHER respondents in {country} are willing to contribute at least 1% of their household income every month to fight global warming.

Note: You are estimating people's BELIEFS about others' willingness, not the actual willingness itself.

Respond with a single number between 0 and 100 with one decimal place."""

    prompts['country_only'] = f"""Country: {country}

{survey_methodology}

{task_minimal}"""
    
    # ================================================================
    # COUNTERFACTUAL 1: GDP FLIP
    # ================================================================
    
    gdp = row.get('gdp_capita_2021', 0)
    is_rich = gdp > 20000
    cf_gdp_str = "$2,000" if is_rich else "$65,000"
    
    economy_gdp_flipped = f"""Economy:
- GDP per capita (PPP, 2021): {cf_gdp_str}
- Top 1% income share: {fmt_pct(row.get('top1pct_income'))}
- Top 1% wealth share: {fmt_pct(row.get('top1pct_wealth'))}
- Human Development Index (2021): {fmt_num(row.get('hdi_2021'), 3)}"""
    
    prompts['cf_gdp_flip'] = f"""Country: {country}

{demographics_full}

{economy_gdp_flipped}

{climate_data}

{survey_methodology}

{actual_willingness}

{task_instruction}"""
    
    # ================================================================
    # COUNTERFACTUAL 2: NAME-DATA MISMATCH
    # ================================================================
    
    all_countries_sorted = sorted(all_countries_df['countrynew'].unique())
    country_idx = all_countries_sorted.index(country) if country in all_countries_sorted else 0
    
    if is_rich:
        poor_countries = all_countries_df[all_countries_df['gdp_capita_2021'] < 5000]['countrynew'].tolist()
        cf_name = sorted(poor_countries)[country_idx % len(poor_countries)] if poor_countries else "Chad"
    else:
        rich_countries = all_countries_df[all_countries_df['gdp_capita_2021'] > 40000]['countrynew'].tolist()
        cf_name = sorted(rich_countries)[country_idx % len(rich_countries)] if rich_countries else "Norway"
    
    survey_methodology_cf = f"""Survey Methodology:
This data comes from a nationally representative survey with a probability-based sample of approximately 1,000 residents aged 15 and above in {cf_name}.

First Question (Personal Willingness):
Respondents were asked: "Would you be willing to contribute 1% of your household income every month to fight global warming? This would mean that you would contribute 1 for every 100 of this income."
- Response options: Yes, No, (Don't Know), (Refused)
- Note: Don't know and refused were coded as missing data

Second Question (Belief About Others):
Respondents were then asked how many respondents in {cf_name} they think are willing to contribute at least 1% of their household income every month to fight global warming.
- Response format: between 0% and 100%, (Don't know), (Refused)"""

    task_instruction_cf = f"""Your Task:
Based on the country and data provided above, estimate what respondents in {cf_name} on average thought about how many OTHER respondents in {cf_name} are willing to contribute at least 1% of their household income every month to fight global warming.

Note: You are estimating people's BELIEFS about others' willingness, not the actual willingness itself.

Respond with a single number between 0 and 100 with one decimal place."""
    
    prompts['cf_name_mismatch'] = f"""Country: {cf_name}

{demographics_full}

{economy_full}

{climate_data}

{survey_methodology_cf}

{actual_willingness}

{task_instruction_cf}"""
    
    # ================================================================
    # COUNTERFACTUAL 3: WILLINGNESS FLIP
    # ================================================================
    
    own_willingness = row.get('mean_own_willingness', 0)
    is_high_willingness = own_willingness > 50
    
    if is_high_willingness:
        cf_own_main = "15.0%"
        cf_own_less = "10.0%"
    else:
        cf_own_main = "75.0%"
        cf_own_less = "15.0%"
    
    actual_willingness_flipped = f"""Personal Willingness (Actual Survey Data):
- {cf_own_main} said they would contribute 1% of their household income monthly
- {cf_own_less} said they would contribute a smaller amount"""
    
    prompts['cf_willingness_flip'] = f"""Country: {country}

{demographics_full}

{economy_full}

{climate_data}

{survey_methodology}

{actual_willingness_flipped}

{task_instruction}"""
    
    return prompts

# ================================================================
# API Functions
# ================================================================

def call_gpt(prompt, model="gpt-4o-mini", max_retries=3):
    if not OPENAI_API_KEY:
        return None
    client = OpenAI(api_key=OPENAI_API_KEY)
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": SYSTEM_INSTRUCTION},
                    {"role": "user", "content": prompt}
                ],
                temperature=0,
                response_format={"type": "json_object"},
            )
            content = response.choices[0].message.content
            if contains_forbidden_strings(content):
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                    continue
                return None
            return extract_number_0_100(content)
        except Exception:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
    return None

def call_claude(prompt, model="claude-3-5-haiku-20241022", max_retries=5):
    if not CLAUDE_API_KEY:
        return None
    client = anthropic.Anthropic(api_key=CLAUDE_API_KEY)
    for attempt in range(max_retries):
        try:
            response = client.messages.create(
                model=model,
                max_tokens=100,
                temperature=0,
                system=SYSTEM_INSTRUCTION,
                messages=[{"role": "user", "content": prompt}],
                tools=[],
            )
            content = response.content[0].text
            if contains_forbidden_strings(content):
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                    continue
                return None
            return extract_number_0_100(content)
        except anthropic.RateLimitError:
            wait_time = (2 ** attempt) * 2
            if attempt < max_retries - 1:
                time.sleep(wait_time)
        except Exception:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
    return None

def call_gemini(prompt, model="gemini-2.5-flash", max_retries=3):
    if not GEMINI_API_KEY:
        return None
    genai.configure(api_key=GEMINI_API_KEY)
    model_obj = genai.GenerativeModel(model)
    full_prompt = f"{SYSTEM_INSTRUCTION}\n\n{prompt}"
    for attempt in range(max_retries):
        try:
            response = model_obj.generate_content(
                full_prompt,
                generation_config=genai.types.GenerationConfig(temperature=0)
            )
            content = response.text
            if contains_forbidden_strings(content):
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                    continue
                return None
            return extract_number_0_100(content)
        except Exception:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
    return None

def call_llama(prompt, model="meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8", max_retries=3):
    if not LLAMA_API_KEY:
        return None
    client = Together(api_key=LLAMA_API_KEY)
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": SYSTEM_INSTRUCTION},
                    {"role": "user", "content": prompt}
                ],
                temperature=0,
            )
            content = response.choices[0].message.content
            if contains_forbidden_strings(content):
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                    continue
                return None
            return extract_number_0_100(content)
        except Exception:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
    return None

# ================================================================
# Identify Missing
# ================================================================

print("\n2. Identifying missing predictions...")

expected = []
for _, row in df.iterrows():
    for condition in condition_order:
        for model_name in MODELS.keys():
            expected.append({
                'country': row['countrynew'],
                'condition': condition,
                'model': model_name
            })

expected_df = pd.DataFrame(expected)
print(f"   ✓ Expected: {len(expected_df)} predictions")

merged = expected_df.merge(
    results_df,
    on=['country', 'condition', 'model'],
    how='left',
    indicator=True
)

missing = merged[merged['_merge'] == 'left_only'][['country', 'condition', 'model']]
print(f"   ✓ Missing: {len(missing)} predictions")

if len(missing) == 0:
    print("\n✓ No missing predictions! Dataset is complete.")
    print(f"Saving complete dataset to: {OUTPUT_FILE}")
    results_df.to_csv(OUTPUT_FILE, index=False)
    exit(0)

print(f"\n3. Filling {len(missing)} missing predictions...")

# Fill missing
new_results = []

for i, (idx, miss) in enumerate(missing.iterrows()):
    country = miss['country']
    condition = miss['condition']
    model_name = miss['model']
    
    country_row = df[df['countrynew'] == country].iloc[0]
    
    # Build ALL prompts
    all_prompts = build_ablated_prompts_structured(country_row, df)
    prompt = all_prompts.get(condition, '')
    
    if not prompt:
        print(f"[{i+1}/{len(missing)}] {country:20s} | {condition:20s} | {model_name:7s} ✗ No prompt")
        continue
    
    print(f"[{i+1}/{len(missing)}] {country:20s} | {condition:20s} | {model_name:7s} ", end="", flush=True)
    
    if model_name == "gpt":
        prediction = call_gpt(prompt, MODELS[model_name])
    elif model_name == "claude":
        prediction = call_claude(prompt, MODELS[model_name])
    elif model_name == "gemini":
        prediction = call_gemini(prompt, MODELS[model_name])
    elif model_name == "llama":
        prediction = call_llama(prompt, MODELS[model_name])
    else:
        prediction = None
    
    if prediction is not None:
        print(f"✓ {prediction:.1f}")
    else:
        print(f"✗ Failed")
    
    new_results.append({
        'country': country,
        'condition': condition,
        'model': model_name,
        'prediction': prediction,
        'actual_other_willingness': country_row.get('mean_other_willingness'),
        'actual_own_willingness': country_row.get('mean_own_willingness'),
    })
    
    if (i + 1) % 10 == 0:
        temp_df = pd.DataFrame(new_results)
        temp_updated = pd.concat([results_df, temp_df], ignore_index=True)
        temp_updated.to_csv(OUTPUT_FILE, index=False)
        print(f"  💾 Progress saved ({len(temp_updated)} total rows)\n")

# Final save
new_df = pd.DataFrame(new_results)
updated_df = pd.concat([results_df, new_df], ignore_index=True)
updated_df.to_csv(OUTPUT_FILE, index=False)

print("\n" + "="*80)
print("FILLING COMPLETE!")
print("="*80)
print(f"Complete dataset saved to: {OUTPUT_FILE}")
print(f"Total predictions: {len(updated_df)}")

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=feb9f195-de2a-416f-b8f1-09efca4e954f' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>